# `ssa_relaxation_budd`

> Generated from `config/ais_0.1_SSA.py` by `config/make_step_notebooks.py`. Edit the driver and regenerate; changes made here are overwritten.

20-yr SSA relaxation. Downstream SSA steps deliberately start from the inverted geometry.

**Reads** (`load_model` calls in this step):
- `f'{model_dir}/AIS3_SSA_inverted.nc'`

**Writes** (`save_model` calls in this step):
- `f'{model_dir}/AIS3_SSA_relaxed.nc'`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_ais_0.1_SSA.pbs`):

```bash
qsub -q normal -l ncpus=8,mem=190GB,walltime=25:00:00 -v NB=ais_0.1_SSA/01_ssa_relaxation_budd.ipynb launch_notebook.pbs
```

The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: none.

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

### What this step does
A 20-year SSA transient with zero SMB and zero melt, time step 0.05 yr. It saves
`AIS3_SSA_relaxed.nc`.

**The relaxed geometry is deliberately not carried forward.** The saved file keeps the
inverted geometry in `md.geometry`; the relaxation itself is only in `md.results`. Later SSA
steps use the file as their model shell. This is by design (worklog §4z correction): only the
HO track adopts its relaxed state.

### Before you run
- `AIS3_SSA_inverted.nc` must exist, with its thickness already restored.
- It takes about 9.5 h. Archive the existing output first (`mv -n`).

### What to check afterwards
- Mean and max |dH|. With the corrected thickness: mean 26.19 m, max about 1.5 km.

### Known pitfalls
- The 400-step history makes the file large. Later steps strip `TransientSolution` right
  after loading it to avoid running out of memory.

_Hand-written notes: `config/notebooks/notes/ssa_relaxation_budd.md` (edit there, then regenerate)._

## Setup

The preamble of `ais_0.1_SSA.py`: imports, helpers, settings and the data catalog.

#### Imports

In [ ]:
"""ACCESS-AIS3 pure-SSA production pipeline (Budd/Weertman friction law, p=q=1 -- the best
validated RMSE anywhere in this project, 60.4 m/yr, vs. the Schoof-based HO track's current
94-99 m/yr after several chunked-restart iterations). See docs/inversion_worklog.md for the
full history of how p=q=1 was found, and the "in your opinion" exchange in this session's
transcript for why a dedicated SSA production track is worth having alongside the HO one:
SSA is dramatically cheaper and already converged/stable, at the cost of no vertical-shear
physics (so it can't do thermal-coupled work the way the HO track can).

Deliberately self-contained -- does NOT import or exec ais_0.1.py, and does NOT touch its
shared `steps`/`save` globals. This project hit two real race-condition incidents this
session from toggling those globals while a job was mid-flight; a fully separate file with
its own local `steps`/`save` sidesteps that risk entirely, at the cost of some code
duplication (the handful of helper functions and Budd-specific step logic below are copied
from ais_0.1.py, not imported, so the two files can diverge safely and be run concurrently).

Friction source: models/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001/ -- the real,
production-canonical location the RMSE=60.4 p=q=1 result was salvaged into (see
config/finalize_p1q1_reg_lcurve.py) from the ad-hoc script that originally produced it
(p1q1_reg_sweep2.py, no longer present as a file, only as raw execution output). This
finalization step recovers the ACTUAL computed result rather than re-deriving it from
scratch, which would otherwise require regenerating AIS3_param.nc with Budd config (it
currently has Schoof baked in, for the HO track) and re-running two expensive inversion
stages. Run finalize_p1q1_reg_lcurve.py once before this script's first stage.

Stage status (updated 2026-09-02):
  1. ssa_inverted_solve_budd -- FULLY IMPLEMENTED AND RUN (RMSE=65.67 m/yr full-continental,
     the best full-continental result in the project). AIS3_SSA_inverted.nc.
  2. ssa_relaxation_budd -- FULLY IMPLEMENTED AND RUN (mean |dH|=32.90m, max |dH|=1505.39m
     over the 20yr relaxation). AIS3_SSA_relaxed.nc.
  3. melt_gamma_tuning_ssa -- builds AIS3_melt_final_ssa.nc directly from the validated
     calibration (gamma_0=300 + per-basin refit deltaT_basin, hard-coded; 2026-09-29). The
     values come from the standalone refit chain (ssa_melt_deltaT_basin_refit.py /
     ssa_melt_deltaT_refit_recalibrate.py). The existing AIS3_melt_final_ssa.nc was built by
     finalize_ssa_melt_calibration.py. The step's old coarse gamma_0 sweep (pick 5537.7,
     AIS3_melt_gamma_tuning_ssa.nc) was superseded by that refit and removed.
  4. historical_dhdt_tuning_ssa -- RUN AND VALIDATED. Full 1995-2019 (24yr) transient against
     AIS3_melt_final_ssa.nc, checked against MIPKIT's dhdt_cpom: grounded RMSE=1.402 m/yr,
     independently reconfirmed via check_historical_ssa.py. AIS3_historical_1995_2019_SSA.nc.
  5. projection_ssp_ssa -- IMPLEMENTED (2026-09-29): anomaly forcing on the calibrated
     baseline (TF0 = Zhou, SMB0 = RACMO 1995-2014), CTRL/SSP/attribution/melt variants chosen
     per job via qsub -v, self-chaining 50-yr chunks to 2300. See launch_projection_ssa.pbs and
     docs/projection_experiments.md.

Both new stages use the plain `cluster` (48 cores/190GB/normal) config, not hugemem -- the
SSA mesh is ~15x smaller than the HO track's (no vertical layers), and both stages use the
same synchronous submit-and-wait pattern (`waitonlock` minutes + `load_only=False` in one
call) already validated throughout this pipeline, avoiding the "never actually submits" bug
found and fixed in ais_0.1.py's own ho_relaxation/historical_dhdt_tuning on their first real
run.
"""
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import glob
import sys
from pathlib import Path
from scipy.spatial import Delaunay

# Site settings (paths, NCI project, ISSM build): edit config/site.env, not this file. Found
# from this script's folder, or from the working directory upwards in a generated notebook.
_here = Path(globals().get('__file__', Path.cwd() / '_')).resolve().parent
sys.path[:0] = [str(p) for p in (_here, *_here.parents) if (p / 'ais3_site.py').exists()][:1]
import ais3_site as site

os.chdir(site.ROOT)
os.environ['ISSM_DIR'] = site.ISSM_DIR

model_dir = f'{site.ROOT}/models'
execution_dir = f'{site.ROOT}/execution_SSA'

# Melt calibration for the whole SSA chain (worklog 6.29-6.31). '' (default) = production: gamma_0=300
# with per-basin refit deltaT. '_hm<gamma_0>' = member hm<gamma_0> of models/melt_calibration_hm.json
# (history-matched calibration, melt_history_match.py + melt_calibration_hm_refit.py), e.g. _hm4000.
# The tag is appended to the melt file, the historical run, the projection initial state and baseline
# and every projection experiment name, so no production file is read as input or overwritten.
SSA_CALIB = os.environ.get('AIS3_SSA_CALIB', '')
assert SSA_CALIB == '' or SSA_CALIB.startswith('_hm'), f'AIS3_SSA_CALIB={SSA_CALIB!r}'
# '_hm<g>sem1' (member of models/melt_calibration_hm_sem1.json, melt_calibration_hm_refit_sem1.py): melt on partly
# floating elements, groundingline.melt_interpolation = SubelementMelt1 in the historical run and every projection
# chunk. Needs the ISSM build with the ismip6 IsAllFloating test removed (Element::Ismip6FloatingiceMeltingRate):
# AIS3_ISSM_DIR must hold the marker file SEM1_PATCH. (Unpatched, the ismip6 class sets no melt on partly floating
# elements, so SubelementMelt1 would silently change nothing.)
CALIB_SEM1 = SSA_CALIB.endswith('sem1')
SEM1_MARKER = f'{site.ISSM_DIR}/SEM1_PATCH'
if CALIB_SEM1:
    assert os.path.exists(SEM1_MARKER), f'AIS3_SSA_CALIB={SSA_CALIB} needs the patched ISSM build ({SEM1_MARKER} missing)'
# Friction-law variant for historical_dhdt_tuning_ssa (worklog 6.32): '' = production Budd p=q=1 with the
# prescribed (Ehrenfeucht) N; p<P>_fixedN / p<P>_flotN = exponent p=P (q=1) with the prescribed N
# (coupling 3) or N from flotation (coupling 2), C rescaled once at the start so the initial basal drag
# is unchanged (rescale_friction_drag_preserving). Appended as _<variant> to the historical run's name.
SSA_FRICTION = os.environ.get('AIS3_SSA_FRICTION', '')
assert SSA_FRICTION in ('', 'p1_flotN', 'p3_fixedN', 'p3_flotN', 'p5_flotN'), f'AIS3_SSA_FRICTION={SSA_FRICTION!r}'
FRIC_TAG = f'_{SSA_FRICTION}' if SSA_FRICTION else ''
# Time-varying ocean forcing for historical_dhdt_tuning_ssa (worklog 6.34): '' = the static Zhou climatology;
# main / cold / warm / vary = ISMIP7 contemporary (OCX) pathway, v2, applied as an anomaly on the calibration
# climatology: TF(t) = TF_Zhou + (OCX_<pathway>(t) - OCX_main(t)), annual columns at year + 0.5. After 1995
# 'main' equals the climatology (so it reproduces the static run); 'vary' carries the observed-like 16-yr
# thermocline oscillation in the Amundsen, Getz and Aurora basins. Appended as _ocx<pathway>.
SSA_OCEAN = os.environ.get('AIS3_SSA_OCEAN', '')
assert SSA_OCEAN in ('', 'main', 'cold', 'warm', 'vary'), f'AIS3_SSA_OCEAN={SSA_OCEAN!r}'
OCEAN_TAG = f'_ocx{SSA_OCEAN}' if SSA_OCEAN else ''
# Prescribed observed ice front for historical_dhdt_tuning_ssa (worklog 6.37): 'greene' = moving front
# (ismovingfront=1, zero calving rate and frontal melt) with spclevelset fixed at every vertex from the
# Greene et al. (2022) masks in MIPKIT (icemask_greene, 24 epochs 1997.75-2021.2). Only observed CHANGE is
# imposed: floating model ice is removed at an epoch if it is ice in the first (1997) mask and not ice at that
# epoch (re-advance allowed); model ice outside the 1997 mask is left alone (dataset mismatch, not change).
# AIS3_HIST_END (default 2019) shortens the run for smoke tests. Tags _frontgreene and _to<end>.
SSA_FRONT = os.environ.get('AIS3_SSA_FRONT', '')
assert SSA_FRONT in ('', 'greene'), f'AIS3_SSA_FRONT={SSA_FRONT!r}'
HIST_END = float(os.environ.get('AIS3_HIST_END', '2019'))
FRONT_TAG = (f'_front{SSA_FRONT}' if SSA_FRONT else '') + (f'_to{HIST_END:.0f}' if HIST_END != 2019 else '')

#### `set_greene_front()` — Moving front prescribed from the observed Greene masks (observed retreat of floating ice only)

In [ ]:
def set_greene_front(md, t_end):
    """Moving front prescribed from the observed Greene masks (observed retreat of floating ice only)."""
    mip = xr.open_dataset(f'{site.ISMIP_DIR}/ISMIP7/AIS/obs/mipkit/AntarcticaObsISMIP7-v1.2.nc')
    gm = mip['icemask_greene']
    gx, gy = mip['x'].values.astype(float), mip['y'].values.astype(float)
    mx = np.asarray(md.mesh.x, dtype=float).ravel()
    my = np.asarray(md.mesh.y, dtype=float).ravel()
    ix = np.clip(np.rint((mx - gx[0]) / (gx[1] - gx[0])).astype(int), 0, gx.size - 1)
    iy = np.clip(np.rint((my - gy[0]) / (gy[1] - gy[0])).astype(int), 0, gy.size - 1)
    tt = mip['greene_mask_time'].values
    tdec = np.array([t.astype('datetime64[D]').astype(object).year +
                     (t.astype('datetime64[D]').astype(object).timetuple().tm_yday - 1) / 365.25 for t in tt])
    il0 = np.asarray(md.mask.ice_levelset, dtype=float).ravel()
    ol0 = np.asarray(md.mask.ocean_levelset, dtype=float).ravel()
    cand = (il0 < 0) & (ol0 < 0)
    g_first = gm.isel(greene_mask_time=0).values[iy, ix] == 1
    cand &= g_first
    nv = md.mesh.numberofvertices
    use = [k for k in range(tdec.size) if tdec[k] <= t_end + 1.0]
    cols = [np.append(il0, float(md.timestepping.start_time))]
    for k in use:
        gone = cand & (gm.isel(greene_mask_time=k).values[iy, ix] == 0)
        cols.append(np.append(np.where(gone, 1.0, il0), tdec[k]))
        print(f"   Greene front {tdec[k]:.2f}: {int(gone.sum())} floating model vertices beyond the observed front", flush=True)
    md.levelset.spclevelset = np.stack(cols, axis=1)
    md.transient.ismovingfront = 1
    md.calving = pyissm.model.classes.calving.default(md.calving)
    md.calving.calvingrate = np.zeros(nv)
    md.frontalforcings = pyissm.model.classes.frontalforcings.default(md.frontalforcings)
    md.frontalforcings.meltingrate = np.zeros(nv)
    md.levelset.kill_icebergs = 1

#### `set_ocx_ocean_forcing()` — Replace the static TF columns with annual TF0 + (OCX pathway - OCX main) columns for `years`

In [ ]:
def set_ocx_ocean_forcing(md, pathway, years):
    """Replace the static TF columns with annual TF0 + (OCX pathway - OCX main) columns for `years`."""
    from scipy.interpolate import RegularGridInterpolator
    root = f'{site.ISMIP_DIR}/ISMIP7/AIS/OCX/ocean'
    dp = xr.open_dataset(f'{root}/{pathway}/v2/tf_AIS_OCX_ocean_{pathway}_v2_1950-2025.nc')['tf']
    dm = xr.open_dataset(f'{root}/main/v2/tf_AIS_OCX_ocean_main_v2_1950-2025.nc')['tf']
    gx, gy = dp['x'].values.astype(float), dp['y'].values.astype(float)
    assert np.allclose(np.abs(dp['z'].values), np.abs(np.asarray(md.basalforcings.tf_depths, dtype=float))), 'OCX depth levels differ from TF0'
    nv = md.mesh.numberofvertices
    pts = np.column_stack([np.asarray(md.mesh.y, dtype=float).ravel(), np.asarray(md.mesh.x, dtype=float).ravel()])
    tf0 = np.stack([np.asarray(t, dtype=float)[:nv, 0] for t in md.basalforcings.tf])     # (nz, nv)
    cols = np.empty((tf0.shape[0], nv + 1, len(years)))
    fl = np.asarray(md.mask.ocean_levelset).ravel() < 0
    for j, yr in enumerate(years):
        sp = dp.sel(time=dp['time.year'] == yr).isel(time=0).to_numpy()
        sm = dm.sel(time=dm['time.year'] == yr).isel(time=0).to_numpy()
        for k in range(tf0.shape[0]):
            a = RegularGridInterpolator((gy, gx), sp[k] - sm[k], method='linear', bounds_error=False, fill_value=np.nan)(pts)
            cols[k, :nv, j] = tf0[k] + np.nan_to_num(a, nan=0.0)
        cols[:, nv, j] = yr + 0.5
        d = cols[:, :nv, j] - tf0
        print(f"   OCX {pathway} {yr}: anomaly vs climatology over floating vertices, all depths: mean {d[:, fl].mean():+.3f} C, "
              f"min {d[:, fl].min():+.3f}, max {d[:, fl].max():+.3f}", flush=True)
    md.basalforcings.tf = [cols[k] for k in range(tf0.shape[0])]

#### `rescale_friction_drag_preserving()` — Budd p=q=1, coupling 3 -> exponent p (q=1), N prescribed or from flotation, same drag at u0

In [ ]:
def rescale_friction_drag_preserving(md, variant):
    """Budd p=q=1, coupling 3 -> exponent p (q=1), N prescribed or from flotation, same drag at u0.

    ISSM Budd drag: tau = C^2 N^(q/p) u^(1/p). Equal drag at the current speed u0 (m/s, floored at
    1 m/yr) gives C_new = C_old * sqrt(N_old * u0^(1 - 1/p) / N_new^(1/p)), as projection_ssp_ssa's
    coupledN / coupledN_p3. N_old is the prescribed N floored at limit*rho_i*g*H; N_new is that, or for
    *_flotN the flotation N (rho_i g H - rho_w g max(-base, 0), same floor) with coupling 2.
    """
    p_new = float(variant[1])
    assert int(md.friction.coupling) == 3, md.friction.coupling
    assert np.all(np.asarray(md.friction.p) == 1) and np.all(np.asarray(md.friction.q) == 1)
    g_, ri_, rw_ = float(md.constants.g), float(md.materials.rho_ice), float(md.materials.rho_water)
    H_ = np.asarray(md.geometry.thickness, dtype=float).ravel()
    base_ = np.asarray(md.geometry.base, dtype=float).ravel()
    floor_ = float(md.friction.effective_pressure_limit) * ri_ * g_ * H_
    N_old = np.maximum(np.maximum(np.asarray(md.friction.effective_pressure, dtype=float).ravel(), 0.0), floor_)
    N_new = np.maximum(ri_ * g_ * H_ - rw_ * g_ * np.maximum(-base_, 0.0), floor_) if variant.endswith('flotN') else N_old
    u0_ = np.maximum(np.asarray(md.initialization.vel, dtype=float).ravel(), 1.0) / float(md.constants.yts)
    ratio = np.sqrt(N_old * u0_ ** (1.0 - 1.0 / p_new) / np.maximum(N_new, 1e-6) ** (1.0 / p_new))
    md.friction.coefficient = np.asarray(md.friction.coefficient, dtype=float).ravel() * ratio
    md.friction.p = p_new * np.ones_like(np.asarray(md.friction.p, dtype=float))
    md.friction.q = np.ones_like(np.asarray(md.friction.q, dtype=float))
    if variant.endswith('flotN'):
        md.friction.coupling = 2
    gr_ = (np.asarray(md.mask.ocean_levelset).ravel() > 0) & (np.asarray(md.mask.ice_levelset).ravel() < 0)
    print(f"-- Friction {variant}: p={p_new:g}, coupling {int(md.friction.coupling)}; grounded N_new/N_old median "
          f"{np.median(N_new[gr_] / N_old[gr_]):.3f}; C factor median {np.median(ratio[gr_]):.3f} "
          f"(p5 {np.percentile(ratio[gr_], 5):.3f}, p95 {np.percentile(ratio[gr_], 95):.3f})")

#### Settings

In [ ]:
save = True
diagnostics = True
plot = True

cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR'] + '/bin'
cluster.executionpath = execution_dir
cluster.storage = site.STORAGE
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']
cluster.np = 48
cluster.memory = 190
cluster.time = 60 * 48
cluster.login = site.USER
cluster.project = site.PROJECT

FRICTION_RUN_DIR = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001'
FRICTION_RUN_NAME = 'run_001_10_100_0.0001'

all_steps = ['ssa_inverted_solve_budd', 'ssa_relaxation_budd', 'melt_gamma_tuning_ssa', 'historical_dhdt_tuning_ssa', 'projection_ssp_ssa']
steps = ['historical_dhdt_tuning_ssa']  # edit before each qsub -- see launch_ais_0.1_SSA.pbs
# Per-job override via `qsub -v AIS3_STEPS=a,b` (same as ais_0.1.py) -- lets a launcher pick its
# step without editing the shared `steps` above.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')

#### `_interp_weights()` — Precompute a reusable linear-interpolation transform (Delaunay triangulation +

In [ ]:
def _interp_weights(src_x, src_y, tgt_x, tgt_y):
    """Precompute a reusable linear-interpolation transform (Delaunay triangulation +
    barycentric weights) ONCE for a fixed (source-grid, target-mesh) pair, so many
    different value arrays defined on that same source grid can each be interpolated in
    milliseconds afterward via _interp_apply.

    Needed for projection_ssp_ssa's time-varying ocean thermal forcing: 30 depths x ~81
    years means ~2430 interpolations of the SAME 761x761 source grid onto the SAME mesh --
    pyissm.data.interp.points_to_mesh (scipy.interpolate.griddata under the hood) rebuilds
    the full Delaunay triangulation AND redoes point-location for every single call, which
    is fine for this pipeline's existing handful-of-calls loops (RACMO's 24 annual SMB
    calls) but empirically verified to cost ~13-20s PER CALL even with triangulation reuse
    alone (evaluation/point-location dominates, not triangulation) -- at 2430 calls that is
    hours just for TF preprocessing, dwarfing the actual transient solve. Precomputing the
    per-target-point simplex + barycentric weights ONCE instead drops each subsequent
    interpolation to ~10ms (verified empirically), since applying already-known weights to
    a new value array is just a weighted sum, not a fresh geometric search.
    """
    src_xy = np.column_stack([np.asarray(src_x).ravel(), np.asarray(src_y).ravel()])
    tgt_xy = np.column_stack([np.asarray(tgt_x).ravel(), np.asarray(tgt_y).ravel()])
    tri = Delaunay(src_xy)
    simplex = tri.find_simplex(tgt_xy)
    vertices = np.take(tri.simplices, simplex, axis=0)
    temp = np.take(tri.transform, simplex, axis=0)
    delta = tgt_xy - temp[:, 2]
    bary = np.einsum('njk,nk->nj', temp[:, :2, :], delta)
    wts = np.hstack((bary, 1 - bary.sum(axis=1, keepdims=True)))
    return vertices, wts, simplex

#### `_interp_apply()` — Apply a transform from _interp_weights to a new value array defined on the same

In [ ]:
def _interp_apply(values, vtx, wts, simplex, fill_value=np.nan):
    """Apply a transform from _interp_weights to a new value array defined on the same
    source grid it was built from -- same interpolation this pipeline uses everywhere else
    (linear, NaN outside the source grid's convex hull), just without repeating the
    triangulation/point-location work each time."""
    values = np.asarray(values).ravel()
    out = np.einsum('nj,nj->n', values[vtx], wts)
    out[simplex == -1] = fill_value
    return out

#### `load_shelf_rheology_B()` — Same as ais_0.1.py's helper -- copied, not imported, deliberately (see module docstring)

In [ ]:
def load_shelf_rheology_B():
    """Same as ais_0.1.py's helper -- copied, not imported, deliberately (see module docstring).
    Floating-ice rheology B from the shelf rheology inversion, independent of friction law:
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17/ (the ais_0.1.py
    rheology_lcurve_run). Until 2026-11-16 this loaded execution_newB_rheology/run_001_1_10_1e-17,
    which finalize_newB_rheology.py saved there unchanged."""
    run = 'run_004_1_10_1e-17'
    mr = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/{run}/{run}.nc')
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Step `ssa_relaxation_budd`

In [ ]:
steps = ['ssa_relaxation_budd']  # this notebook's step; overrides the driver default and AIS3_STEPS

In [ ]:
print("-------------------------------------------------------------")
print(" TRANSIENT RELAXATION (PURE-SSA BUDD)")
print("-------------------------------------------------------------")

#### Loading inverted model

In [ ]:
print("-- Loading inverted model...")
md = pyissm.model.io.load_model(f'{model_dir}/AIS3_SSA_inverted.nc')

md.inversion.iscontrol = 0
md.verbose.solution = 1

md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
md.transient.isstressbalance = 1
md.transient.ismasstransport = 1
md.transient.issmb = 1
md.transient.isthermal = 0
md.transient.isgroundingline = 1
md.groundingline.migration = 'SubelementMigration'
md.transient.requested_outputs = ['default', 'Vel', 'Thickness', 'Surface', 'Base', 'MaskOceanLevelset']

# Same NaN-default/zero-placeholder fixes already found and documented in ais_0.1.py's
# ssa_relaxation -- this is a fresh transient activation on a freshly-assembled model, so
# they apply again here.
md.masstransport.spcthickness = np.full(md.mesh.numberofvertices, np.nan)
md.smb.mass_balance = np.zeros(md.mesh.numberofvertices)
md.basalforcings.groundedice_melting_rate = np.zeros(md.mesh.numberofvertices)
md.basalforcings.floatingice_melting_rate = np.zeros(md.mesh.numberofvertices)

# TODO: tune final_time/time_step for your actual use case, same as ais_0.1.py's own
# ssa_relaxation -- 20yr/0.05yr is a placeholder shock-damping window, not a validated
# choice for this specific track.
md.timestepping.start_time = 0
md.timestepping.final_time = 20
md.timestepping.time_step = 0.05

#### Assigning cluster and updating settings

In [ ]:
print("-- Assigning cluster and updating settings...")
md.miscellaneous.name = 'AIS3_SSA_relaxed'
md.cluster = cluster
md.settings.waitonlock = 1440
md.settings.solver_residue_threshold = 1e-3

if save:
    print("-- Submitting and waiting on transient relaxation...")
    md = pyissm.model.execute.solve(md, 'Transient', load_only=False, runtime_name=False)

    if diagnostics:
        dH = np.asarray(md.results.TransientSolution[-1].Thickness).ravel() - np.asarray(md.geometry.thickness).ravel()
        print(f"\nRELAXATION DIAGNOSTICS:")
        print(f"   Max |dH| over relaxation: {np.nanmax(np.abs(dH)):.2f} m")
        print(f"   Mean |dH| over relaxation: {np.nanmean(np.abs(dH)):.2f} m")

    print(f"\nSaving to {model_dir}/AIS3_SSA_relaxed.nc")
    pyissm.model.io.save_model(md, f'{model_dir}/AIS3_SSA_relaxed.nc')
else:
    print("-- Submitting transient relaxation...")
    md = pyissm.model.execute.solve(md, 'Transient', load_only=False, runtime_name=False)